In [1]:
import numpy as np 
from tqdm import tqdm
import cv2
import os
import tensorflow as tf
#import tensorflow_hub as hub
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
#import itertools
#import plotly.graph_objs as go
#from plotly.offline import init_notebook_mode, iplot
#from plotly import tools
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPool2D
#from keras.preprocessing.image import ImageDataGenerator
from keras.applications.vgg16 import VGG16, preprocess_input
from keras import layers
from keras.models import Model, Sequential
from keras.optimizers import Adam, RMSprop
from keras.callbacks import EarlyStopping
#from keras.preprocessing.image import ImageDataGenerator
#init_notebook_mode(connected=True)
RANDOM_SEED = 123

2024-12-21 14:52:06.702285: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-12-21 14:52:10.589028: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-21 14:52:12.179383: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-21 14:52:12.783560: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-21 14:52:15.897699: I tensorflow/core/platform/cpu_feature_guar

In [2]:
TRAIN_DIR = ('/mnt/c/Users/tobia/datasets/emotion3/archive(5)/Grayscale Face images/train/')
TEST_DIR = ('/mnt/c/Users/tobia/datasets/emotion3/archive(5)/Grayscale Face images/test/')
VAL_DIR = ('/mnt/c/Users/tobia/datasets/emotion3/archive(5)/Grayscale Face images/validation/')


In [3]:
IMG_SIZE = (48,48)
def load_data(dir_path, IMG_SIZE):
    
    X = []
    y = []
    i = 0
    labels = dict()
    for path in tqdm(sorted(os.listdir(dir_path))):
        if not path.startswith('.'):
            labels[i] = path
            for file in os.listdir(dir_path + path):
                if not file.startswith('.'):
                    img = cv2.imread(dir_path + path + '/' + file)
                    img = img.astype('float32') / 255
                    resized = cv2.resize(img, IMG_SIZE, interpolation = cv2.INTER_AREA)
                    X.append(resized)
                    y.append(i)
            i += 1
    X = np.array(X)
    y = np.array(y)
    print(f'{len(X)} images loaded from {dir_path} directory.')
    return X, y, labels

train_X, train_y, labels = load_data(TRAIN_DIR, IMG_SIZE)
val_X, val_y, labels = load_data(VAL_DIR, IMG_SIZE)
test_X, test_y, labels = load_data(TEST_DIR, IMG_SIZE)

100%|████████████████████████████████████████████████████████████████████████████████████| 8/8 [21:19<00:00, 159.91s/it]


90043 images loaded from /mnt/c/Users/tobia/datasets/emotion3/archive(5)/Grayscale Face images/train/ directory.


100%|█████████████████████████████████████████████████████████████████████████████████████| 8/8 [04:22<00:00, 32.76s/it]


19292 images loaded from /mnt/c/Users/tobia/datasets/emotion3/archive(5)/Grayscale Face images/validation/ directory.


100%|█████████████████████████████████████████████████████████████████████████████████████| 8/8 [04:44<00:00, 35.55s/it]


19302 images loaded from /mnt/c/Users/tobia/datasets/emotion3/archive(5)/Grayscale Face images/test/ directory.


In [4]:
from keras.utils import to_categorical

train_Y = to_categorical(train_y, num_classes=8)
print(train_Y.shape)

(90043, 8)


In [5]:
from keras.applications.vgg19 import VGG19
import math
base_model = VGG19(
        weights=None,
        include_top=False, 
        input_shape=IMG_SIZE + (1, )
    )

base_model.summary()

#from keras.applications.EfficientNetB3 import EfficientNetB3
#base_model = keras.applications.EfficientNetB3(
#    weights = None, include_top = False, input_shape = IMG_SIZE
#)



NUM_CLASSES = 8

model = Sequential()
model.add(base_model)
model.add(Flatten())
model.add(Dense(1000, activation="relu"))
model.add(Dropout(0.1))
model.add(Dense(NUM_CLASSES, activation="softmax"))

model.compile(
    loss='binary_crossentropy',
    optimizer=RMSprop(learning_rate=1e-4),
    metrics=['accuracy'])




I0000 00:00:1734797527.701389     517 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1734797533.086387     517 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1734797533.086715     517 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1734797533.102953     517 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1734797533.103026     517 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:0

Model: "vgg19"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 48, 48, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 48, 48, 64)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 48, 48, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 24, 24, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 24, 24, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 24, 24, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 12, 12, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 12, 12, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 12, 12, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv4 (Conv2D)           │ (None, 12, 12, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 6, 6, 512)      │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 6, 6, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 6, 6, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv4 (Conv2D)           │ (None, 6, 6, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 3, 3, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 3, 3, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 3, 3, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 3, 3, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv4 (Conv2D)           │ (None, 3, 3, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 1, 1, 512)      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,023,232 (76.38 MB)

 Trainable params: 20,023,232 (76.38 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
epochs = 100
batch_size = 128
history = model.fit(train_X, train_Y, epochs=epochs,
                    validation_data = (val_X, val_y), batch_size = batch_size, verbose=1)

In [ ]:
TRAIN_WRITE_DIR = ('/mnt/c/Users/tobia/datasets/emotionData2/train')
dataset = tf.data.Dataset.from_tensor_slices((X, Y))
dataset = train_set.shuffle(len(dataset), seed=42)
#valid_set = tf.data.Dataset.from_tensor_slices((X_valid, y_valid))




def write_images(WRITE_DIR, X, y):
    y_write = []
    for path in tqdm(sorted(os.listdir(dir_path))):
        if not file.startswith('.'):
            for index in range(len(X)):
                cv2.imwrite(X[index])
                y_write.append(y[index])
write_images(WRITE_DIR, X, y)                     